In [1]:
# Cell 1 — Setup
import sys
for mod in list(sys.modules.keys()):
    if mod in ('config','data_loader','preprocessor',
               'trainer','evaluator','tuner'):
        del sys.modules[mod]
sys.path.insert(0, '../src')

import warnings, joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from data_loader import (load_raw_data, clean_dependents,
                          handle_credit_history, encode_target,
                          get_X_y, get_train_val_split)
from trainer     import train_all_models
from evaluator   import evaluate_all_models
from tuner       import tune_model
from config      import MODELS_DIR

df, _ = load_raw_data()
df    = clean_dependents(df)
df    = handle_credit_history(df)
df    = encode_target(df)
X, y  = get_X_y(df)
X_train, X_val, y_train, y_val = get_train_val_split(X, y)

# Base pipelines for comparison
base_pipelines = train_all_models(X_train, y_train)
base_results   = evaluate_all_models(base_pipelines, X_val, y_val)
print("Base results loaded.")

  Training LogisticRegression... done
  Training DecisionTree... done
  Training RandomForest... done
  Training GradientBoosting... done
  Training XGBoost... done
Base results loaded.


In [2]:
# Cell 2 — Tune Random Forest (~3 minutes)
rf_tuned, rf_params, rf_cv_score = tune_model(
    "RandomForest", X_train, y_train, n_iter=60
)
joblib.dump(rf_tuned, MODELS_DIR / "RandomForest_tuned.joblib")
print("\nRandom Forest tuned model saved.")


Tuning RandomForest (60 iterations × 5 folds = 300 model fits)...
Fitting 5 folds for each of 60 candidates, totalling 300 fits

  Best CV ROC-AUC : 0.7641
  Best params:
    n_estimators              = 100
    min_samples_split         = 10
    min_samples_leaf          = 4
    max_features              = 0.5
    max_depth                 = 10

Random Forest tuned model saved.


In [3]:
# Cell 3 — Tune XGBoost (~5 minutes)
xgb_tuned, xgb_params, xgb_cv_score = tune_model(
    "XGBoost", X_train, y_train, n_iter=60
)
joblib.dump(xgb_tuned, MODELS_DIR / "XGBoost_tuned.joblib")
print("\nXGBoost tuned model saved.")


Tuning XGBoost (60 iterations × 5 folds = 300 model fits)...
Fitting 5 folds for each of 60 candidates, totalling 300 fits

  Best CV ROC-AUC : 0.7615
  Best params:
    subsample                 = 0.8
    reg_lambda                = 1
    reg_alpha                 = 0.1
    n_estimators              = 200
    min_child_weight          = 5
    max_depth                 = 4
    learning_rate             = 0.05
    colsample_bytree          = 0.8

XGBoost tuned model saved.


In [4]:
# Cell 4 — Before vs After comparison
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

tuned_pipelines = {
    "RandomForest_tuned": rf_tuned,
    "XGBoost_tuned":      xgb_tuned,
}

rows = []
# Base models — RF and XGB only for clean comparison
for name in ["RandomForest", "XGBoost"]:
    pipe   = base_pipelines[name]
    y_pred = pipe.predict(X_val)
    y_prob = pipe.predict_proba(X_val)[:, 1]
    rows.append({
        "model":     name + "_base",
        "roc_auc":   round(roc_auc_score(y_val, y_prob),  4),
        "f1":        round(f1_score(y_val, y_pred),        4),
        "precision": round(precision_score(y_val, y_pred), 4),
        "recall":    round(recall_score(y_val, y_pred),    4),
    })

# Tuned models
for name, pipe in tuned_pipelines.items():
    y_pred = pipe.predict(X_val)
    y_prob = pipe.predict_proba(X_val)[:, 1]
    rows.append({
        "model":     name,
        "roc_auc":   round(roc_auc_score(y_val, y_prob),  4),
        "f1":        round(f1_score(y_val, y_pred),        4),
        "precision": round(precision_score(y_val, y_pred), 4),
        "recall":    round(recall_score(y_val, y_pred),    4),
    })

comparison = pd.DataFrame(rows).set_index("model")
print("\n── Before vs After Tuning ──")
print(comparison.to_string())


── Before vs After Tuning ──
                    roc_auc      f1  precision  recall
model                                                 
RandomForest_base    0.8214  0.8950     0.8438  0.9529
XGBoost_base         0.7731  0.8539     0.8172  0.8941
RandomForest_tuned   0.8096  0.9071     0.8469  0.9765
XGBoost_tuned        0.7858  0.8827     0.8404  0.9294


In [5]:
# Cell 5 — Verify overfit gap reduced on tuned models
from sklearn.model_selection import cross_val_score, StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in [("RF_base",  base_pipelines["RandomForest"]),
                    ("RF_tuned", rf_tuned),
                    ("XGB_base", base_pipelines["XGBoost"]),
                    ("XGB_tuned",xgb_tuned)]:
    scores = cross_val_score(pipe, X_train, y_train,
                              cv=skf, scoring='roc_auc', n_jobs=-1)
    print(f"{name:<12}  CV AUC = {scores.mean():.4f} ± {scores.std():.4f}")

RF_base       CV AUC = 0.7659 ± 0.0757
RF_tuned      CV AUC = 0.7641 ± 0.0783
XGB_base      CV AUC = 0.7422 ± 0.0876
XGB_tuned     CV AUC = 0.7615 ± 0.0749


In [6]:
# notebooks/04_hyperparameter_tuning.ipynb — add this cell

# FINAL MODEL SELECTION — documented decision
# ─────────────────────────────────────────────────────────────────────────────
# Candidate       Val AUC    CV AUC    CV Std    Decision
# RF_base         0.8214     0.7659    0.0757    High variance, lucky split
# RF_tuned        0.8096     0.7641    0.0783    Similar CV, slightly worse std
# XGB_base        0.7731     0.7422    0.0876    Undertuned
# XGB_tuned       0.7858     0.7615    0.0749    Best CV + lowest std on XGB
#
# WINNER: RandomForest_base for deployment
# Reasoning: Highest CV AUC (0.7659) with acceptable std.
#            Single-split AUC 0.8214 — most representative of held-out perf.
#            Simpler than tuned XGB, no risk of meta-overfitting.
#            With only 614 rows, simplicity and stability beat marginal gains.
# ─────────────────────────────────────────────────────────────────────────────

import joblib
from config import MODELS_DIR

FINAL_MODEL_NAME = "RandomForest"
final_pipeline   = base_pipelines["RandomForest"]   # already trained

# Save as the canonical production model
joblib.dump(final_pipeline, MODELS_DIR / "final_model.joblib")
print(f"Final model saved: {MODELS_DIR / 'final_model.joblib'}")
print(f"Model type: {type(final_pipeline.named_steps['classifier']).__name__}")
print(f"Val AUC:    0.8214")
print(f"CV AUC:     0.7659 ± 0.0757")
print(f"Threshold:  0.77 (cost-optimized)")

Final model saved: c:\Users\Shresth\OneDrive\Desktop\loan-eligibility-predictor\notebooks\..\models\final_model.joblib
Model type: RandomForestClassifier
Val AUC:    0.8214
CV AUC:     0.7659 ± 0.0757
Threshold:  0.77 (cost-optimized)
